In [ ]:
import time
import logging
from functools import wraps
from dataclasses import dataclass
from datetime import datetime
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
import tushare as ts

# ==========================================
# 0. 配置中心
# ==========================================
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')
logger = logging.getLogger("IndexPricing")

@dataclass
class PricingConfig:
    target_month: str      # '2609'
    delivery_date: str     # '20260918'
    spot_code: str = '000905.SH'  # 中证500
    threshold: float = 0.005      # 50BP 偏离度触发阈值
    fallback_dy: float = 0.021    # 容灾股息率 (中证500历史均值附近)

    @property
    def fut_code(self) -> str:
        prefix = "IC" if self.spot_code == '000905.SH' else "IM"
        return f"{prefix}{self.target_month}.CFX"

pro = ts.pro_api()

def with_retry(max_retries=3, delay=2.0):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_retries - 1: raise e
                    time.sleep(delay * (2 ** attempt))
        return wrapper
    return decorator

# ==========================================
# 1. 数据核心 (无期权依赖)
# ==========================================
@with_retry()
def get_latest_trade_date():
    today = datetime.now().strftime('%Y%m%d')
    cal = pro.trade_cal(exchange='SSE', start_date='20260101', end_date=today, is_open='1')
    return cal['cal_date'].max()

@with_retry()
def fetch_market_data(trade_date, cfg):
    spot_df = pro.index_daily(ts_code=cfg.spot_code, start_date=trade_date, end_date=trade_date)
    fut_df = pro.fut_daily(ts_code=cfg.fut_code, start_date=trade_date, end_date=trade_date)
    return spot_df['close'].iloc[0], fut_df['close'].iloc[0]

@with_retry()
def fetch_shibor_rate(trade_date, days):
    shibor_df = pro.shibor(start_date=trade_date, end_date=trade_date)
    row = shibor_df.iloc[0]
    nodes = np.array([1, 7, 14, 30, 90, 180, 270, 360])
    rates = np.array([row['on'], row['1w'], row['2w'], row['1m'], row['3m'], row['6m'], row['9m'], row['1y']]) / 100.0
    return float(interp1d(nodes, rates, kind='linear', fill_value="extrapolate")(days))

@with_retry()
def fetch_constituent_q(trade_date, cfg):
    """合成 500 只成分股的加权 TTM 股息率"""
    # 1. 获取权重
    w_df = pro.index_weight(index_code=cfg.spot_code, start_date='20260101', end_date=trade_date)
    w_df = w_df[w_df['trade_date'] == w_df['trade_date'].max()].rename(columns={'con_code': 'ts_code'})
    
    # 2. 获取个股基本面
    b_df = pro.daily_basic(trade_date=trade_date, fields='ts_code,dv_ttm')
    
    # 3. 合并计算
    m = pd.merge(w_df, b_df, on='ts_code', how='inner')
    if m.empty: return cfg.fallback_dy
    
    m['w_norm'] = m['weight'] / m['weight'].sum()
    m['dv_clean'] = m['dv_ttm'].fillna(0).clip(upper=15.0) / 100.0
    
    q_final = float((m['dv_clean'] * m['w_norm']).sum())
    return q_final if q_final > 0 else cfg.fallback_dy

# ==========================================
# 2. 测算逻辑
# ==========================================
def main():
    cfg = PricingConfig(target_month='2609', delivery_date='20260918')
    
    try:
        t_day = get_latest_trade_date()
        logger.info(f"=== [成分股驱动版] 定价测算启动 ({t_day}) ===")
        
        # 时间参数
        days = (datetime.strptime(cfg.delivery_date, '%Y%m%d') - datetime.strptime(t_day, '%Y%m%d')).days
        T = days / 365.0
        
        # 抓取数据
        S, F = fetch_market_data(t_day, cfg)
        r = fetch_shibor_rate(t_day, days)
        q = fetch_constituent_q(t_day, cfg)
        
        # 计算 FV (连续复利)
        FV = S * np.exp((r - q) * T)
        dev = (F - FV) / FV
        
        # 输出结果
        print("\n" + "="*45)
        print(f"标的: {cfg.spot_code} ({cfg.fut_code})")
        print(f"现货: {S:.2f} | 期货: {F:.2f} | 剩余: {days}天")
        print(f"参数: r={r:.2%} | q={q:.2%} | FV={FV:.2f}")
        print(f"结论: 偏离度 {dev:.3%} -> {'高估 (空)' if dev > cfg.threshold else '低估 (多)' if dev < -cfg.threshold else '合理'}")
        print("="*45 + "\n")
        
        return t_day, S, r, T, q
        
    except Exception as e:
        logger.error(f"Execution Error: {e}")
        return None, None, None, None, None

# 暴露全局变量给后续 Cell
trade_date, S_0, r, T, q_final = main()

In [ ]:
import pandas as pd
import numpy as np

def extract_detailed_dividend_analysis(trade_date, cfg):
    """
    深入分析中证 500 成份股的分红贡献
    """
    print(f"\n--- 正在执行中证 500 分红深度诊断 ({trade_date}) ---")
    
    # 1. 获取权重 (自动适配 spot_code)
    weight_df = pro.index_weight(index_code=cfg.spot_code, start_date='20260101', end_date=trade_date)
    weight_df = weight_df[weight_df['trade_date'] == weight_df['trade_date'].max()]
    if 'con_code' in weight_df.columns:
        weight_df = weight_df.rename(columns={'con_code': 'ts_code'})
    
    # 2. 获取个股基本面 (dv_ttm) 和 行业分类
    basic_df = pro.daily_basic(trade_date=trade_date, fields='ts_code,dv_ttm')
    # 额外获取股票列表以获得行业信息
    stock_info = pro.stock_basic(fields='ts_code,symbol,name,industry')
    
    # 3. 数据合并
    merged = pd.merge(weight_df, basic_df, on='ts_code', how='inner')
    merged = pd.merge(merged, stock_info, on='ts_code', how='left')
    
    # 4. 计算贡献度
    merged['weight_norm'] = merged['weight'] / merged['weight'].sum()
    # 计算每只股票贡献的 BP (1 BP = 0.01%)
    merged['contribution_bp'] = (merged['dv_ttm'] / 100.0) * merged['weight_norm'] * 10000
    
    # 5. 输出统计摘要
    print(f"有效样本数: {len(merged)} 只")
    
    # A. 个股贡献前 10
    print(f"\n[TOP 50 个股贡献榜 (BP)]")
    display(merged.sort_values('contribution_bp', ascending=False).head(50)[
        ['ts_code', 'name', 'industry', 'weight', 'dv_ttm', 'contribution_bp']
    ])
    
    # B. 行业贡献汇总
    industry_sum = merged.groupby('industry')['contribution_bp'].sum().sort_values(ascending=False)
    print(f"\n[行业贡献汇总 (BP)]")
    display(industry_sum.head(10))
    
    # 6. 保存到文件
    output_file = f"csi500_dividend_deep_dive_{trade_date}.xlsx"
    merged.sort_values('contribution_bp', ascending=False).to_excel(output_file, index=False)
    print(f"\n数据已保存至: {output_file}")

# ==========================================
# 执行诊断 (引用方案 1 暴露的全局变量)
# ==========================================
if 'trade_date' in globals() and trade_date is not None:
    # 自动引用主程序的配置
    try:
        # 确保配置类存在
        cfg_diag = PricingConfig(target_month='2609', delivery_date='20260918')
        extract_detailed_dividend_analysis(trade_date, cfg_diag)
    except Exception as e:
        print(f"诊断执行失败: {e}")
else:
    print("错误：请先运行主程序 Cell 以获取 trade_date 和 market 数据。")